# Device fingerprints under controlled conditions

`recording_fingerprint.ipynb` could not test the "project every clip onto a common fingerprint" idea fairly, because
in KweriMoto each source is also a different set of vehicles. Here the fingerprint is **simulated**, so the ground
truth is known: the same clips are passed through several artificial "devices", and nothing else changes.

Three questions:

1. **Frozen foundation models** — how much device information do AST and CLAP embeddings carry, and does a cheap
   input projection (spectral equalisation) remove it without retraining?
2. **Fault cues** — does the projection keep the information needed to detect faults? And are features built on
   rhythm rather than tone immune to the device by construction?
3. **Benchmark audit** — were KweriMoto's clips already in the pre-trained model's training data?

Only the train and validation splits are used. Classifiers are logistic regressions on frozen features.

## 1. Setup and data

In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio.compliance.kaldi as kaldi
import torchaudio.functional as AF
from huggingface_hub import snapshot_download
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import ASTFeatureExtractor, ASTModel, ClapModel, ClapProcessor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "kwerimoto"
RESULTS_DIR = ROOT / "results"
CACHE_DIR = RESULTS_DIR / "cache"
FIG_DIR = RESULTS_DIR / "figures"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

AST_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
CLAP_ID = "laion/clap-htsat-unfused"
SR = 16_000
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

snapshot_download("rayyyyy007/kwerimoto-engine-sounds", repo_type="dataset", local_dir=DATA_DIR)
full_manifest = pd.read_csv(DATA_DIR / "manifest.csv")

# Two single-source tasks, so the real recording source never differs between the classes being compared.
TASKS = {"knock (Revix)": ("revix", "engine_knock"), "misfire (HF)": ("huggingface", "misfire")}
keep = full_manifest["source_dataset"].isin(["revix", "huggingface"]) & full_manifest["split"].isin(["train", "validation"])
manifest = full_manifest[keep].reset_index(drop=True)
waveforms = [sf.read(DATA_DIR / f, dtype="float32")[0] for f in manifest["file_name"]]
is_train = (manifest["split"] == "train").to_numpy()
print(manifest.groupby(["source_dataset", "label", "split"]).size().unstack())

## 2. Simulated devices and the projection

Each device is a fixed processing chain applied to a clip. Two are **fixed filters** (the textbook model of a
microphone's tone colour); three are not, and are included to see where the fixed-filter assumption breaks.

| Device | What it does | Fixed filter? |
|---|---|---|
| `reference` | nothing | — |
| `phone` | cuts bass, boosts 3 kHz, rolls off the top | yes |
| `muffled` | boosts bass, steadily cuts treble | yes |
| `noisy` | adds white noise at 15 dB signal-to-noise ratio | no |
| `mp3` | MP3 encode/decode at 16 kbit/s | no |
| `compressor` | strong dynamic-range compression | no |

**Projection (`eq`):** rescale each frequency so the clip's long-term average spectrum equals one common reference
(the average over the original training clips). It is applied to the waveform, so it works with any model, and it
needs no knowledge of which device made the recording.

In [ ]:
N_FFT, HOP = 512, 160
WINDOW = torch.hann_window(N_FFT)
FREQS = np.linspace(0, SR / 2, N_FFT // 2 + 1)


def stft(w):
    return torch.stft(torch.from_numpy(w), N_FFT, HOP, window=WINDOW, return_complex=True)


def istft(S, length):
    return torch.istft(S, N_FFT, HOP, window=WINDOW, length=length).numpy()


def filter_device(points_db):
    """Device defined by a fixed frequency response, given as (Hz, dB) control points."""
    hz, db = zip(*points_db)
    gain = torch.tensor(10 ** (np.interp(FREQS, hz, db) / 20), dtype=torch.float32)[:, None]
    return lambda w, i: istft(stft(w) * gain, len(w))


def noisy(w, i, snr_db=15):
    noise = np.random.default_rng(SEED + i).standard_normal(len(w)).astype(np.float32)
    return w + noise * np.sqrt((w ** 2).mean() / 10 ** (snr_db / 10))


def mp3(w, i, bitrate="16k"):
    pcm = ["-f", "f32le", "-ar", str(SR), "-ac", "1"]
    run = lambda args, data: subprocess.run(["ffmpeg", "-v", "error", *args], input=data, capture_output=True, check=True).stdout
    encoded = run([*pcm, "-i", "pipe:0", "-c:a", "libmp3lame", "-b:a", bitrate, "-f", "mp3", "pipe:1"], w.tobytes())
    decoded = np.frombuffer(run(["-f", "mp3", "-i", "pipe:0", *pcm, "pipe:1"], encoded), dtype=np.float32)
    out = np.zeros(len(w), dtype=np.float32)
    out[:min(len(w), len(decoded))] = decoded[:len(w)]
    return out


def compressor(w, i, drive=8.0):
    return (np.tanh(drive * w / (np.abs(w).max() + 1e-8)) / np.tanh(drive) * np.abs(w).max()).astype(np.float32)


DEVICES = {
    "reference": lambda w, i: w,
    "phone": filter_device([(0, -30), (100, -18), (300, -3), (1000, 0), (3000, 6), (5000, 0), (7000, -12), (8000, -20)]),
    "muffled": filter_device([(0, 3), (200, 6), (800, 0), (2000, -6), (4000, -14), (8000, -24)]),
    "noisy": noisy,
    "mp3": mp3,
    "compressor": compressor,
}
FIXED_FILTERS = ["phone", "muffled"]


def average_spectrum(w):
    """Long-term average log power spectrum of a clip (one value per FFT frequency)."""
    return torch.log(stft(w).abs().pow(2).mean(1) + 1e-10)


REFERENCE = torch.stack([average_spectrum(w) for w, t in zip(waveforms, is_train) if t]).mean(0)
MAX_GAIN = np.log(10 ** (40 / 20))  # limit correction to +-40 dB so near-silent bands are not blown up


def equalise(w):
    gain = (0.5 * (REFERENCE - average_spectrum(w))).clamp(-MAX_GAIN, MAX_GAIN).exp()[:, None]
    return istft(stft(w) * gain, len(w))

In [ ]:
example = int(np.flatnonzero((manifest["source_dataset"] == "revix") & (manifest["duration_s"] > 9))[0])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for name, device in DEVICES.items():
    processed = device(waveforms[example], example)
    axes[0].plot(FREQS, 10 / np.log(10) * average_spectrum(processed), label=name, lw=1)
    axes[1].plot(FREQS, 10 / np.log(10) * average_spectrum(equalise(processed)), label=name, lw=1)
axes[0].set(title="One clip through each device: average spectrum", xlabel="frequency (Hz)", ylabel="level (dB)")
axes[1].set(title="Same clips after the eq projection", xlabel="frequency (Hz)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "devices_spectra.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Features

Every clip × device × {raw, eq} version is turned into four kinds of features:

- **AST** and **CLAP** — frozen foundation-model embeddings.
- **avg spectrum** — the clip's average level in 40 frequency bands. A positive control: by construction it carries
  fixed-filter fingerprints.
- **rhythm** — the modulation spectrum: how strongly each frequency region pulses at rates from 0.5 to 50 Hz, after
  removing each band's average level. Removing the average makes it blind to fixed filters *by construction*, while
  it keeps the pulse patterns that a misfire or knock should produce.

In [ ]:
ast_extractor = ASTFeatureExtractor.from_pretrained(AST_ID)
ast = ASTModel.from_pretrained(AST_ID).to(DEVICE).eval()
clap_processor = ClapProcessor.from_pretrained(CLAP_ID)
clap = ClapModel.from_pretrained(CLAP_ID).to(DEVICE).eval()

MOD_FRAMES = 1024
mod_hz = np.fft.rfftfreq(MOD_FRAMES, d=0.01)
mod_edges = np.geomspace(0.5, 50, 17)


def log_mel(w):
    return kaldi.fbank(torch.from_numpy(w)[None] * 32768, num_mel_bins=40, sample_frequency=SR, dither=0.0).numpy()


def rhythm_features(spec):
    centred = spec - spec.mean(0)                                             # removes any fixed filter
    looped = np.tile(centred, (MOD_FRAMES // len(spec) + 1, 1))[:MOD_FRAMES]
    modulation = np.abs(np.fft.rfft(looped, axis=0))                          # (modulation rate, band)
    rates = np.stack([modulation[(mod_hz >= lo) & (mod_hz < hi)].mean(0) for lo, hi in zip(mod_edges, mod_edges[1:])])
    return np.log1p(rates.reshape(len(rates), 8, 5).mean(2)).ravel()          # 16 rates x 8 band groups


@torch.inference_mode()
def extract(audios):
    out = {"AST": [], "CLAP": []}
    for i in range(0, len(audios), 32):
        batch = ast_extractor(audios[i:i + 32], sampling_rate=SR, return_tensors="pt")
        out["AST"].append(ast(batch["input_values"].to(DEVICE)).pooler_output.float().cpu().numpy())
    for i in range(0, len(audios), 16):
        audio_48k = [AF.resample(torch.from_numpy(w), SR, 48_000).numpy() for w in audios[i:i + 16]]
        batch = clap_processor(audio=audio_48k, sampling_rate=48_000, return_tensors="pt")
        out["CLAP"].append(clap.get_audio_features(**{k: v.to(DEVICE) for k, v in batch.items()}).pooler_output.float().cpu().numpy())
    specs = [log_mel(w) for w in audios]
    return {"AST": np.concatenate(out["AST"]), "CLAP": np.concatenate(out["CLAP"]),
            "avg spectrum": np.stack([s.mean(0) for s in specs]),
            "rhythm": np.stack([rhythm_features(s) for s in specs])}


MODELS = ["AST", "CLAP", "avg spectrum", "rhythm"]
features = {}  # (device, "raw" | "eq") -> {model: array of shape (clips, dims)}
for name, device in DEVICES.items():
    paths = {v: CACHE_DIR / f"devices_{name}_{v}.npz" for v in ("raw", "eq")}
    if not all(p.exists() for p in paths.values()):
        processed = [np.ascontiguousarray(device(w, i), dtype=np.float32) for i, w in enumerate(waveforms)]
        np.savez(paths["raw"], **extract(processed))
        np.savez(paths["eq"], **extract([equalise(w) for w in processed]))
    for v, p in paths.items():
        features[name, v] = dict(np.load(p))
    print(name, "done")

## 4. Question 1 — how visible is each device, and does `eq` hide it?

For each device, half of the clips (chosen at random, independent of the label) are passed through that device and
half through the reference. A classifier then tries to tell which is which. **0.5 = invisible, 1.0 = obvious.**

In [ ]:
def fit_predict(F_train, t_train, F_test):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    return clf.fit(F_train, t_train).predict_proba(F_test)[:, 1]


def mix(model, version, device, use_device):
    """Per-clip features: from `device` where use_device is True, otherwise from the reference device."""
    return np.where(use_device[:, None], features[device, version][model], features["reference", version][model])


assigned = np.random.default_rng(SEED).random(len(manifest)) < 0.5
visibility = pd.DataFrame({
    (model, version): {
        device: balanced_accuracy_score(
            assigned[~is_train],
            fit_predict(mix(model, version, device, assigned)[is_train], assigned[is_train],
                        mix(model, version, device, assigned)[~is_train]) > 0.5)
        for device in DEVICES if device != "reference"}
    for model in MODELS for version in ("raw", "eq")
})
visibility.to_csv(RESULTS_DIR / "devices_visibility.csv")
visibility.round(2)

## 5. Questions 1 and 2 — a planted shortcut

This recreates KweriMoto's problem with a known cause. In **training**, every healthy clip comes from the reference
device and every faulty clip from device *d*, so "device" predicts the label perfectly. The model is then tested
with the devices **swapped** (healthy from *d*, faulty from the reference):

- a model that learned the *fault* still scores high;
- a model that learned the *device* scores **below 0.5**, because the cue now points the wrong way.

The **clean** row is the same task with every clip from the reference device — the best that can be expected, and
the check that `eq` does not destroy fault cues (question 2).

In [ ]:
rows = []
for task, (src, fault_label) in TASKS.items():
    in_task = (manifest["source_dataset"] == src).to_numpy()
    fault = (manifest["label"] == fault_label).to_numpy()
    train, test = in_task & is_train, in_task & ~is_train
    for model in MODELS:
        for version in ("raw", "eq"):
            clean = features["reference", version][model]
            rows.append({"task": task, "model": model, "version": version, "device": "clean",
                         "swapped": roc_auc_score(fault[test], fit_predict(clean[train], fault[train], clean[test]))})
            for device in DEVICES:
                if device == "reference":
                    continue
                score = fit_predict(mix(model, version, device, fault)[train], fault[train],
                                    mix(model, version, device, ~fault)[test])
                rows.append({"task": task, "model": model, "version": version, "device": device,
                             "swapped": roc_auc_score(fault[test], score)})
shortcut = pd.DataFrame(rows)
shortcut.to_csv(RESULTS_DIR / "devices_shortcut.csv", index=False)

order = ["clean", *[d for d in DEVICES if d != "reference"]]
table = shortcut.pivot_table(index=["task", "device"], columns=["model", "version"], values="swapped")
table = table.reindex(columns=[(m, v) for m in MODELS for v in ("raw", "eq")])
table = table.reindex([(t, d) for t in TASKS for d in order])
table.round(2)

In [ ]:
fig, axes = plt.subplots(len(TASKS), len(MODELS), figsize=(15, 6), sharey=True)
x = np.arange(len(order) - 1)
for r, task in enumerate(TASKS):
    for c, model in enumerate(MODELS):
        ax = axes[r, c]
        for offset, version, color in [(-0.2, "raw", "tab:red"), (0.2, "eq", "tab:blue")]:
            ax.bar(x + offset, table.loc[task].loc[order[1:], (model, version)], 0.4, color=color, label=version)
            ax.axhline(table.loc[(task, "clean"), (model, version)], color=color, ls="--", lw=1)
        ax.axhline(0.5, color="grey", ls=":", lw=1)
        ax.set(title=f"{model} — {task}", ylim=(0, 1.02))
        ax.set_xticks(x, order[1:], rotation=30, ha="right")
axes[0, 0].legend(title="bars: swapped-device AUC\ndashed: clean AUC", fontsize=8, title_fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("AUC with devices swapped")
plt.tight_layout()
plt.savefig(FIG_DIR / "devices_shortcut.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Question 3 — were KweriMoto's clips in AST's pre-training data?

AST was trained with labels on AudioSet's training lists (balanced + unbalanced) and evaluated on its evaluation
list. AudioSet publishes these lists as YouTube IDs, so we can check each KweriMoto clip that names a YouTube ID.

In [ ]:
META_DIR = ROOT / "data" / "audioset_meta"
META_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv"
LISTS = {"unbalanced_train": "unbalanced_train_segments.csv", "balanced_train": "balanced_train_segments.csv",
         "eval": "eval_segments.csv"}

for filename in [*LISTS.values(), "class_labels_indices.csv"]:
    if not (META_DIR / filename).exists():
        subprocess.run(["curl", "-sSfL", f"{BASE_URL}/{filename}", "-o", str(META_DIR / filename)], check=True)

classes = pd.read_csv(META_DIR / "class_labels_indices.csv")
knock_id = classes.loc[classes["display_name"] == "Engine knocking", "mid"].item()

audioset = {}  # YouTube ID -> (list name, labelled "Engine knocking"?)
for list_name, filename in LISTS.items():
    segments = pd.read_csv(META_DIR / filename, comment="#", header=None, skipinitialspace=True,
                           names=["ytid", "start", "end", "labels"])
    for ytid, labels in zip(segments["ytid"], segments["labels"]):
        audioset[ytid] = (list_name, knock_id in labels.split(","))
print(f"{len(audioset):,} AudioSet segments; 'Engine knocking' = {knock_id}")

In [ ]:
segment_id = lambda filename: filename.rsplit("_", 1)[0] if isinstance(filename, str) else None

clip_sets = {
    "AudioSet knock clips": full_manifest[full_manifest["source_dataset"] == "audioset"]
        .assign(ytid=lambda d: d["original_file"].map(segment_id)),
    "Revix base recordings": full_manifest[full_manifest["source_dataset"] == "revix"]
        .assign(ytid=lambda d: d["source_recording"].map(segment_id)),
    "Revix overlaid knock sounds": full_manifest[full_manifest["synthetic"]]
        .assign(ytid=lambda d: d["knock_source"].map(segment_id)),
}
audit = []
for name, clips in clip_sets.items():
    found = clips["ytid"].map(lambda v: audioset.get(v, ("not in AudioSet", False)))
    clips = clips.assign(audioset_list=found.str[0], knock_label=found.str[1])
    counts = clips.groupby(["split", "audioset_list"]).size().unstack(fill_value=0)
    counts["labelled 'Engine knocking'"] = clips.groupby("split")["knock_label"].sum()
    audit.append(counts.assign(clips=name).set_index("clips", append=True).swaplevel())
audit = pd.concat(audit).fillna(0).astype(int)
audit.to_csv(RESULTS_DIR / "audioset_contamination.csv")
audit